# Fine-tune YOLO11: từ best.pt 2 lớp sang dataset 3 lớp

Notebook này **khởi tạo từ trọng số `best.pt` cũ**, không phải `resume` phiên train cũ. Khi số lớp đổi từ 2 sang 3, tầng phát hiện cuối được điều chỉnh cho 3 lớp; các đặc trưng học được trước đó vẫn là điểm khởi đầu. Độ chính xác có thể tăng hoặc giảm: cần đối chứng bằng tập test cố định và chỉ số từng lớp.

**Trước khi chạy:** tải `backend/models/best.pt` của dự án lên `MyDrive/Colab/models/best_2classes.pt`. Chạy notebook Prepare_Helmet_Dataset_3Classes_Clean trước để tạo `MyDrive/Colab/helmet_dataset_3classes_clean.zip`, rồi chạy các cell train theo thứ tự trên Colab GPU. Không ghi đè file gốc.

Dataset do notebook Prepare_Helmet_Dataset_3Classes tạo có nhãn bbox và polygon trộn trong một số file. Notebook này chuyển polygon thành bbox **chỉ trong bản giải nén tạm trên Colab** trước khi train detection; ZIP trên Drive không bị sửa.


## 1. Cấu hình lần train


In [ ]:
from pathlib import Path

TRAIN_NAME = "finetune_best_2_to_3_lan_1"  # đổi tên nếu chạy lần khác
ZIP_PATH = Path("/content/drive/MyDrive/Colab/helmet_dataset_3classes_clean.zip")
OLD_WEIGHTS = Path("/content/drive/MyDrive/Colab/models/best_2classes.pt")
PROJECT_DIR = Path("/content/drive/MyDrive/Colab/YOLO11_Helmet_Independent_Runs")

EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 20
WORKERS = 2
SEED = 42
LEARNING_RATE = 0.001  # thấp hơn mức mặc định để fine-tune ổn định hơn; có thể thử nghiệm
TRAIN_EXTRA_ARGS = {}

RUN_REAL_TEST = False
REAL_DATA_DIR = Path("/content/drive/MyDrive/Colab/du_lieu_thuc_te")
REAL_TEST_CONF = 0.25

TRAIN_DIR = PROJECT_DIR / TRAIN_NAME
EXPORT_DIR = TRAIN_DIR / "bao_cao_export"
print("Trọng số gốc:", OLD_WEIGHTS)
print("Dataset:", ZIP_PATH)
print("Kết quả:", TRAIN_DIR)


## 2. Kiểm tra GPU và mount Google Drive


In [ ]:
import torch
from google.colab import drive

drive.mount("/content/drive")

print("GPU khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Tên GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("⚠️ Chưa bật GPU. Vào Runtime → Change runtime type → T4 GPU.")


## 3. Giải nén và kiểm tra dataset


In [ ]:
from zipfile import ZipFile
import shutil
import yaml
import math

DATASET_DIR = Path("/content/helmet_dataset_3classes")
if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy dataset: {ZIP_PATH}")

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)  # chỉ xóa bản giải nén tạm trong runtime Colab
DATASET_DIR.mkdir(parents=True)
with ZipFile(ZIP_PATH) as z:
    z.extractall(DATASET_DIR)

yaml_files = list(DATASET_DIR.rglob("data.yaml"))
if len(yaml_files) != 1:
    raise ValueError(f"Cần đúng một data.yaml, tìm thấy: {yaml_files}")
SOURCE_YAML = yaml_files[0]
DATASET_ROOT = SOURCE_YAML.parent
with SOURCE_YAML.open(encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

names = data_cfg.get("names")
if isinstance(names, dict):
    names = [names[i] if i in names else names[str(i)] for i in range(len(names))]
if not isinstance(names, list) or len(names) != 3 or len(set(map(str, names))) != 3:
    raise ValueError(f"Dataset phải có đúng 3 tên lớp khác nhau; hiện có: {names}")
if int(data_cfg.get("nc", 3)) != 3:
    raise ValueError(f"nc không phải 3: {data_cfg.get('nc')}")

def split_path(value):
    if isinstance(value, list):
        raise ValueError("Notebook này cần mỗi split là một đường dẫn thư mục")
    p = Path(value)
    if not p.is_absolute():
        p = DATASET_ROOT / p
    return p.resolve()

split_paths = {}
for split in ("train", "val", "test"):
    if data_cfg.get(split):
        p = split_path(data_cfg[split])
        if not p.is_dir():
            raise FileNotFoundError(f"Thiếu thư mục {split}: {p}. Kiểm tra data.yaml trong ZIP")
        split_paths[split] = p
if "train" not in split_paths or "val" not in split_paths:
    raise ValueError("Dataset cần train và val")

# Tạo YAML riêng cho Colab, không sửa ZIP gốc. Bỏ path cũ từ máy/Roboflow.
runtime_cfg = {"path": str(DATASET_ROOT), "train": str(split_paths["train"]),
               "val": str(split_paths["val"]), "nc": 3, "names": names}
if "test" in split_paths:
    runtime_cfg["test"] = str(split_paths["test"])
DATA_YAML = DATASET_DIR / "data_colab_3classes.yaml"
with DATA_YAML.open("w", encoding="utf-8") as f:
    yaml.safe_dump(runtime_cfg, f, allow_unicode=True, sort_keys=False)
data_cfg = runtime_cfg

# Dataset nguồn trộn bbox và polygon. Chuẩn hóa polygon thành bbox trên bản giải nén tạm.
# Không thay đổi ZIP trên Google Drive.
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
dataset_stats = {}
for split, image_dir in split_paths.items():
    parts = list(image_dir.parts)
    if "images" not in parts:
        raise ValueError(f"Không suy ra được thư mục labels từ {image_dir}")
    parts[parts.index("images")] = "labels"
    label_dir = Path(*parts)
    if not label_dir.is_dir():
        raise FileNotFoundError(f"Thiếu thư mục labels: {label_dir}")
    ids = set()
    format_counts = {"bbox": 0, "polygon": 0}
    converted_files = 0
    label_files = list(label_dir.rglob("*.txt"))
    for label_file in label_files:
        new_lines = []
        converted_here = 0
        for line_no, line in enumerate(label_file.read_text(encoding="utf-8-sig").splitlines(), 1):
            if not line.strip():
                continue
            parts = line.split()
            try:
                class_value = float(parts[0])
                class_id = int(class_value)
                coords = [float(value) for value in parts[1:]]
            except ValueError as exc:
                raise ValueError(f"Nhãn không phải số tại {label_file}:{line_no}") from exc
            if class_value != class_id:
                raise ValueError(f"ID lớp không nguyên tại {label_file}:{line_no}")
            if class_id not in (0, 1, 2):
                raise ValueError(f"ID lớp ngoài 0..2 tại {label_file}:{line_no}")
            if len(parts) == 5:
                format_counts["bbox"] += 1
                new_lines.append(line.strip())
            elif len(parts) >= 7 and len(parts) % 2 == 1:
                if not all(math.isfinite(v) and 0 <= v <= 1 for v in coords):
                    raise ValueError(f"Tọa độ polygon ngoài [0,1] tại {label_file}:{line_no}")
                xs, ys = coords[0::2], coords[1::2]
                x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
                if x2 <= x1 or y2 <= y1:
                    raise ValueError(f"Polygon không có diện tích tại {label_file}:{line_no}")
                box = ((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1)
                new_lines.append(str(class_id) + " " + " ".join(f"{v:.8f}" for v in box))
                format_counts["polygon"] += 1
                converted_here += 1
            else:
                raise ValueError(f"Nhãn {label_file}:{line_no} có {len(parts)} giá trị; cần 5 (bbox) hoặc 7+ số lẻ (polygon). Đầu dòng: {line[:120]!r}")
            ids.add(class_id)
        if converted_here:
            label_file.write_text("\n".join(new_lines) + "\n", encoding="utf-8")
            converted_files += 1
    n_images = sum(p.suffix.lower() in image_exts for p in image_dir.rglob("*"))
    dataset_stats[split] = {"images": n_images, "labels": len(label_files), "class_ids": sorted(ids), "original_label_formats": format_counts, "files_converted": converted_files}
    print(split, dataset_stats[split])
    if converted_files:
        print(f"  Đã chuyển {format_counts['polygon']} polygon thành bbox trong {converted_files} file tạm.")
    if n_images == 0 or not label_files:
        raise ValueError(f"Split {split} không có ảnh hoặc nhãn")
if set(dataset_stats["train"]["class_ids"]) != {0, 1, 2}:
    raise ValueError("Tập train chưa chứa đủ cả 3 lớp")
print("Thứ tự lớp mới:", dict(enumerate(names)))


In [ ]:
# Thống kê số ảnh và số file nhãn theo từng tập
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def resolve_split_path(value):
    if value is None:
        return None
    p = Path(value)
    if not p.is_absolute():
        p = DATASET_ROOT / p
    return p

def count_files(folder, extensions=None):
    if folder is None or not folder.exists():
        return 0
    files = [p for p in folder.rglob("*") if p.is_file()]
    if extensions:
        files = [p for p in files if p.suffix.lower() in extensions]
    return len(files)

dataset_stats = dict(globals().get("dataset_stats", {}))
for split in ["train", "val", "test"]:
    image_path = resolve_split_path(data_cfg.get(split))
    image_count = count_files(image_path, IMAGE_EXTS)

    label_path = None
    if image_path is not None:
        # Cấu trúc YOLO phổ biến: images/... và labels/...
        parts = list(image_path.parts)
        if "images" in parts:
            idx = parts.index("images")
            parts[idx] = "labels"
            label_path = Path(*parts)
        else:
            label_path = image_path.parent / "labels"

    label_count = count_files(label_path, {".txt"})
    dataset_stats[split] = {
        **dataset_stats.get(split, {}),
        "images": image_count,
        "labels": label_count,
        "image_path": str(image_path) if image_path else None
    }
    print(f"{split:5s}: {image_count} ảnh | {label_count} nhãn")


## 4. Cài đặt Ultralytics


In [ ]:
!pip install -q -U ultralytics pyyaml

import ultralytics
ultralytics.checks()


## 5. Train YOLO11


In [ ]:
from ultralytics import YOLO
import time
from datetime import datetime
import hashlib

if not OLD_WEIGHTS.is_file():
    raise FileNotFoundError(f"Hãy upload best.pt cũ lên Drive: {OLD_WEIGHTS}")
if TRAIN_DIR.exists():
    raise FileExistsError(f"{TRAIN_DIR} đã tồn tại. Hãy đổi TRAIN_NAME để giữ kết quả cũ.")

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

SOURCE_SHA256 = sha256(OLD_WEIGHTS)
old_model = YOLO(str(OLD_WEIGHTS))
OLD_NAMES = old_model.names
if len(OLD_NAMES) != 2:
    raise ValueError(f"best.pt cũ cần đúng 2 lớp, hiện có: {OLD_NAMES}")
if old_model.task != "detect":
    raise ValueError(f"Cần checkpoint detection, nhận được: {old_model.task}")
print("Lớp cũ:", OLD_NAMES)
print("Lớp mới:", dict(enumerate(names)))
print("SHA256 checkpoint cũ:", SOURCE_SHA256)

# Fine-tune từ checkpoint cũ. Đây là lượt train mới, KHÔNG dùng resume=True.
# Ultralytics sẽ thay tầng dự đoán cho nc=3; trọng số không khớp hình dạng
# ở tầng này không được tái sử dụng, còn phần backbone/neck được chuyển.
DEVICE = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(str(OLD_WEIGHTS))
train_args = dict(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE,
                  batch=BATCH_SIZE, device=DEVICE, project=str(PROJECT_DIR),
                  name=TRAIN_NAME, exist_ok=False, resume=False,
                  pretrained=True, patience=PATIENCE, workers=WORKERS,
                  cache=False, plots=True, save=True, seed=SEED,
                  deterministic=True, lr0=LEARNING_RATE)
train_args.update(TRAIN_EXTRA_ARGS)
if train_args.get("resume"):
    raise ValueError("Không dùng resume khi đổi từ 2 sang 3 lớp")
started_at = datetime.now()
timer = time.perf_counter()
model.train(**train_args)
training_seconds = time.perf_counter() - timer
finished_at = datetime.now()
BEST_MODEL_PATH = TRAIN_DIR / "weights" / "best.pt"
LAST_MODEL_PATH = TRAIN_DIR / "weights" / "last.pt"
if not BEST_MODEL_PATH.is_file():
    raise FileNotFoundError(f"Train chưa tạo best.pt: {BEST_MODEL_PATH}")
best_model = YOLO(str(BEST_MODEL_PATH))
if len(best_model.names) != 3:
    raise RuntimeError(f"Checkpoint mới không có 3 lớp: {best_model.names}")
print("Hoàn tất:", BEST_MODEL_PATH)
print("Lớp checkpoint mới:", best_model.names)
print(f"Thời gian: {training_seconds / 60:.1f} phút")


## 6. Đánh giá mô hình tốt nhất

In [ ]:
from ultralytics import YOLO
import numpy as np
import pandas as pd

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy best.pt tại: {BEST_MODEL_PATH}")

best_model = YOLO(str(BEST_MODEL_PATH))
requested_split = "test" if data_cfg.get("test") else "val"
EVALUATION_DIR = TRAIN_DIR / "danh_gia_best"

metrics = best_model.val(
    data=str(DATA_YAML),
    split=requested_split,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(TRAIN_DIR),
    name="danh_gia_best",
    exist_ok=True,
    plots=True,
)

precision = float(metrics.box.mp)
recall = float(metrics.box.mr)
map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)

f1_values = np.asarray(getattr(metrics.box, "f1", []), dtype=float)
if f1_values.size > 0:
    mean_f1 = float(np.nanmean(f1_values))
else:
    mean_f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

summary_metrics = {
    "precision": precision,
    "recall": recall,
    "f1_score": mean_f1,
    "map50": map50,
    "map50_95": map50_95,
    "fitness": float(metrics.fitness) if metrics.fitness is not None else None,
    "evaluation_split": requested_split,
}

try:
    per_class_df = pd.DataFrame(metrics.summary(decimals=6))
except Exception:
    per_class_df = pd.DataFrame()

print("=" * 55)
print("KẾT QUẢ ĐÁNH GIÁ BEST.PT")
print("=" * 55)
print(f"Tập đánh giá : {requested_split}")
print(f"Precision    : {precision:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"F1-score     : {mean_f1:.4f}")
print(f"mAP@50       : {map50:.4f}")
print(f"mAP@50-95    : {map50_95:.4f}")

print("\nCHỈ SỐ THEO TỪNG LỚP")
display(per_class_df)

## Kiểm thử ảnh và video thực tế từ Google Drive

### Cách thêm dữ liệu

1. Mở **Google Drive**.
2. Tạo thư mục:

```text
MyDrive/Colab/du_lieu_thuc_te/
```

3. Tải vào thư mục đó các ảnh hoặc video **không thuộc tập train, validation và test**.

Ví dụ:

```text
du_lieu_thuc_te/
├── anh_ban_ngay_01.jpg
├── anh_nhieu_xe_02.png
├── video_duong_pho_01.mp4
└── video_ban_dem_02.mp4
```

Bạn cũng có thể chia thành thư mục con:

```text
du_lieu_thuc_te/
├── anh/
│   ├── anh_01.jpg
│   └── anh_02.jpg
└── video/
    ├── video_01.mp4
    └── video_02.mp4
```

4. Quay lại **Bước cấu hình** và đổi:

```python
RUN_REAL_TEST = True
```

5. Chạy cell bên dưới.

Kết quả có khung nhận diện sẽ được lưu vào thư mục `ket_qua_thuc_te` bên trong thư mục của lần train.

> Nên đặt tên tệp khác nhau để tránh trùng tên khi lưu kết quả.

In [ ]:
# Lấy tên lớp trực tiếp từ best.pt
model_names = best_model.names

if isinstance(model_names, dict):
    class_names = list(model_names.values())
else:
    class_names = list(model_names)

frame_count = 0
total_boxes = 0

class_counts = {
    str(class_name): 0
    for class_name in class_names
}

In [ ]:
# ============================================================
# CHỈ CHẠY KHI RUN_REAL_TEST = True
# ============================================================

REAL_IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"
}

REAL_VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv", ".m4v",
    ".mpeg", ".mpg", ".wmv", ".webm", ".ts"
}

REAL_SUPPORTED_EXTENSIONS = REAL_IMAGE_EXTENSIONS | REAL_VIDEO_EXTENSIONS
REAL_RESULT_DIR = TRAIN_DIR / "ket_qua_thuc_te"
REAL_TEST_STATUS = "Chưa chạy"
real_test_rows = []

if not RUN_REAL_TEST:
    print("⏭️ Đã bỏ qua kiểm thử thực tế.")
    print("Sau khi thêm ảnh/video vào Drive, đổi RUN_REAL_TEST = True.")
    print("Thư mục dữ liệu thực tế:")
    print(REAL_DATA_DIR)
else:
    if not REAL_DATA_DIR.exists():
        REAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
        print("⚠️ Đã tạo thư mục:")
        print(REAL_DATA_DIR)
        print("Hãy tải ảnh/video vào thư mục này rồi chạy lại cell.")

    real_files = sorted(
        path
        for path in REAL_DATA_DIR.rglob("*")
        if path.is_file() and path.suffix.lower() in REAL_SUPPORTED_EXTENSIONS
    )

    if not real_files:
        print("⚠️ Chưa tìm thấy ảnh hoặc video hợp lệ trong:")
        print(REAL_DATA_DIR)
    else:
        print(f"Tìm thấy {len(real_files)} tệp thực tế.")

        for file_index, source_file in enumerate(real_files, start=1):
            file_type = (
                "image"
                if source_file.suffix.lower() in REAL_IMAGE_EXTENSIONS
                else "video"
            )

            print(
                f"[{file_index}/{len(real_files)}] "
                f"Đang xử lý: {source_file.name}"
            )

            prediction_stream = best_model.predict(
                source=str(source_file),
                imgsz=IMAGE_SIZE,
                conf=REAL_TEST_CONF,
                device=DEVICE,
                save=True,
                save_txt=True,
                save_conf=True,
                project=str(TRAIN_DIR),
                name="ket_qua_thuc_te",
                exist_ok=True,
                stream=True,
                verbose=False,
            )

            frame_count = 0
            total_boxes = 0
            class_counts = {str(class_name): 0 for class_name in class_names}

            for prediction in prediction_stream:
                frame_count += 1

                if prediction.boxes is None:
                    continue

                predicted_classes = (
                    prediction.boxes.cls.detach().cpu().numpy().astype(int)
                )

                total_boxes += len(predicted_classes)

                for class_id in predicted_classes:
                    class_name = (
                        str(class_names[class_id])
                        if 0 <= class_id < len(class_names)
                        else str(class_id)
                    )
                    class_counts[class_name] = class_counts.get(class_name, 0) + 1

            row = {
                "file": source_file.name,
                "relative_path": str(source_file.relative_to(REAL_DATA_DIR)),
                "type": file_type,
                "frames_processed": frame_count,
                "total_boxes_across_frames": total_boxes,
            }

            for class_name, count in class_counts.items():
                row[f"count_{class_name}"] = count

            real_test_rows.append(row)

        REAL_TEST_STATUS = "Đã chạy"
        real_test_df = pd.DataFrame(real_test_rows)
        REAL_TEST_CSV = TRAIN_DIR / "ket_qua_thuc_te_tong_hop.csv"
        real_test_df.to_csv(REAL_TEST_CSV, index=False, encoding="utf-8-sig")

        print("\n✅ Kiểm thử thực tế hoàn tất")
        print("Ảnh/video có khung nhận diện:")
        print(REAL_RESULT_DIR)
        print("Bảng tổng hợp:")
        print(REAL_TEST_CSV)
        display(real_test_df)

        print(
            "\nLưu ý: với video, tổng số box là tổng lượt xuất hiện "
            "của các box qua các frame, không phải số đối tượng duy nhất."
        )

## 8. Export toàn bộ dữ liệu phục vụ báo cáo

In [ ]:
import json
import csv
import platform
from datetime import datetime

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for model_path in [BEST_MODEL_PATH, LAST_MODEL_PATH]:
    if model_path.exists():
        shutil.copy2(model_path, EXPORT_DIR / model_path.name)

important_patterns = [
    "results.png", "results.csv", "args.yaml",
    "confusion_matrix.png", "confusion_matrix_normalized.png",
    "PR_curve.png", "P_curve.png", "R_curve.png", "F1_curve.png",
    "labels.jpg", "labels_correlogram.jpg",
    "train_batch*.jpg", "val_batch*_labels.jpg", "val_batch*_pred.jpg",
]

for base_dir in [TRAIN_DIR, EVALUATION_DIR]:
    if not base_dir.exists():
        continue
    for pattern in important_patterns:
        for src in base_dir.glob(pattern):
            if src.is_file():
                prefix = "eval_" if base_dir == EVALUATION_DIR else "train_"
                shutil.copy2(src, EXPORT_DIR / f"{prefix}{src.name}")

PER_CLASS_CSV = TRAIN_DIR / "ket_qua_theo_tung_lop.csv"
per_class_df.to_csv(PER_CLASS_CSV, index=False, encoding="utf-8-sig")

report_data = {
    "train_name": TRAIN_NAME,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "method": "Fine-tune từ best.pt 2 lớp sang 3 lớp",
    "source_checkpoint": str(OLD_WEIGHTS),
    "source_sha256": SOURCE_SHA256,
    "source_classes": OLD_NAMES,
    "model": str(OLD_WEIGHTS),
    "epochs_configured": EPOCHS,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "patience": PATIENCE,
    "seed": SEED,
    "learning_rate": LEARNING_RATE,
    "classes": data_cfg.get("names"),
    "number_of_classes": data_cfg.get("nc", len(data_cfg.get("names", []))),
    "dataset": dataset_stats,
    "metrics": summary_metrics,
    "per_class_metrics": per_class_df.to_dict(orient="records"),
    "real_test": {
        "status": REAL_TEST_STATUS,
        "source_directory": str(REAL_DATA_DIR),
        "confidence": REAL_TEST_CONF,
        "files": real_test_rows,
    },
    "training_minutes": training_seconds / 60,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "python": platform.python_version(),
    "ultralytics": ultralytics.__version__,
    "best_model": str(BEST_MODEL_PATH),
}

with open(TRAIN_DIR / "ket_qua_day_du.json", "w", encoding="utf-8") as f:
    json.dump(report_data, f, ensure_ascii=False, indent=2)

with open(TRAIN_DIR / "chi_so_danh_gia.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow([
        "train_name", "precision", "recall", "f1_score",
        "mAP50", "mAP50-95", "split", "training_minutes"
    ])
    writer.writerow([
        TRAIN_NAME,
        precision,
        recall,
        mean_f1,
        map50,
        map50_95,
        requested_split,
        training_seconds / 60,
    ])

report_txt = f"""KẾT QUẢ FINE-TUNE YOLO11 TỪ BEST.PT 2 LỚP - {TRAIN_NAME}
=======================================================
Model gốc           : {OLD_WEIGHTS}
Epoch               : {EPOCHS}
Image size          : {IMAGE_SIZE}
Batch size          : {BATCH_SIZE}
GPU                 : {report_data['gpu']}
Số lớp              : {report_data['number_of_classes']}
Tên lớp             : {report_data['classes']}
Thời gian train     : {training_seconds / 60:.2f} phút

KẾT QUẢ ĐÁNH GIÁ ({requested_split})
Precision           : {precision:.4f}
Recall              : {recall:.4f}
F1-score            : {mean_f1:.4f}
mAP@50              : {map50:.4f}
mAP@50-95           : {map50_95:.4f}

MA TRẬN NHẦM LẪN
- danh_gia_best/confusion_matrix.png
- danh_gia_best/confusion_matrix_normalized.png

KIỂM THỬ THỰC TẾ
Trạng thái          : {REAL_TEST_STATUS}
Dữ liệu đầu vào     : {REAL_DATA_DIR}
Kết quả nhận diện   : {REAL_RESULT_DIR}

Đường dẫn best.pt   : {BEST_MODEL_PATH}
"""

with open(TRAIN_DIR / "bao_cao_ket_qua.txt", "w", encoding="utf-8") as f:
    f.write(report_txt)

print("✅ Export hoàn tất")
print("Thư mục lần train:", TRAIN_DIR)
print("Thư mục tài liệu báo cáo:", EXPORT_DIR)


## 9. Nén thư mục kết quả để tải về máy (tùy chọn)

In [ ]:
archive_path = shutil.make_archive(
    f"/content/{TRAIN_NAME}",
    "zip",
    root_dir=TRAIN_DIR,
)

print("✅ File ZIP:", archive_path)

# Bỏ dấu # nếu muốn tải về máy.
# from google.colab import files
# files.download(archive_path)